# Creation of figures relating to CeLLaTe data stats

Beginning with entity annotations per paper annotated, sourced from Google sheet at https://docs.google.com/spreadsheets/d/1fQpxvOkTRyt0fjywfrj9yeig3DbzNvf8ZrnV1GEGyKg/edit?usp=sharing

In [9]:
# import sys
# !{sys.executable} -m ensurepip --upgrade
# !{sys.executable} -m pip install bokeh


In [23]:
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
output_notebook()


Loading BokehJS ...

In [29]:
import pandas as pd

# Read-in paper stats file
in_file = pd.read_csv("/Users/withers/Downloads/CeLLaTe-Paper-Records - Paper-Stats.csv")
in_file = in_file.drop(['Unnamed: 9'], axis=1)
in_file = in_file[in_file["Paper Source"] != "CheMBL_V1"]
in_file.head()

,PMCID,Paper Source,Entities-Annotated,Total num sentences,Total Sentences with entities,Total Entity Count,Tissue Count,CellType Count,CellLine Count,Paper Source.1,% Sentences w. entities,% Tissue entities,% CellType entities,% CellLine entities,Sanity check
0,PMC12115102,CheMBL_V2,"Celltype, Tissue, Cellline",99.0,35.0,53.0,10.0,1.0,42.0,CheMBL,35.4,18.9,1.9,79.2,100.0
1,PMC12101583,CheMBL_V2,"Celltype, Tissue, Cellline",43.0,21.0,45.0,6.0,15.0,24.0,CheMBL,48.8,13.3,33.3,53.3,99.9
2,PMC12081701,CheMBL_V2,"Celltype, Cellline",101.0,8.0,9.0,0.0,1.0,8.0,CheMBL,7.9,0.0,11.1,88.9,100.0
3,PMC11075828,CheMBL_V2,"Celltype, Tissue, Cellline",78.0,12.0,14.0,1.0,9.0,4.0,CheMBL,15.4,7.1,64.3,28.6,100.0
4,PMC10442740,CheMBL_V2,"Celltype, Tissue, Cellline",38.0,19.0,34.0,10.0,17.0,7.0,CheMBL,50.0,29.4,50.0,20.6,100.0


In [54]:
chembl_df = in_file[in_file['Paper Source'] == 'CheMBL_V2']
sc_df = in_file[in_file['Paper Source'] == 'Single-Cell']
cf_df = in_file[in_file['Paper Source'] == 'CellFinder']


In [101]:
from bokeh.models import ColumnDataSource, FactorRange
from bokeh.plotting import figure, show
from bokeh.transform import factor_cmap
from typing import List, TypeAlias, Optional

bokeh_plot : TypeAlias = figure

palette = ["#55a7e2", "#f3d476", "#33cc33bb"]  # Blue, Orange, Green
# start=1, end=2 tells it to only evaluate the entity_type part of your tuple!
color_mapper = factor_cmap(field_name='x', palette=palette, factors=entity_types, start=1, end=2)

def stacked_barplot(input_df: pd.DataFrame, width: Optional[int] = 800) -> bokeh_plot:

        pmcids = list(input_df['PMCID'])
        entity_types = ['Tissue', 'CellType', 'CellLine'] #Sub-groups of greater PMCID assignment
        df_source = input_df['Paper Source'].iloc[0]
        if df_source == 'CheMBL_V2':
                df_source = 'ChEMBL'
        elif df_source == 'Single-Cell':
                df_source = 'Single Cell'

        x = [(pmcid, entity_type) for pmcid in pmcids for entity_type in entity_types] #Labels

        counts = []
        for i in input_df.index:
                counts.append(input_df.loc[i, 'Tissue Count'])
                counts.append(input_df.loc[i, 'CellType Count'])
                counts.append(input_df.loc[i, 'CellLine Count'])

        source = ColumnDataSource(data=dict(x=x, counts=counts))

        p = figure(y_range=FactorRange(*x), height=800, width=width, title=f"Entity Counts for '{df_source}' sourced articles",
                toolbar_location=None, tools="")

        p.hbar(y='x', right='counts', height=0.9, source=source, fill_color=color_mapper, line_color=color_mapper)

        p.x_range.start = 0
        p.xgrid.grid_line_color = None
        p.ygrid.grid_line_color = None
        p.yaxis.group_label_orientation = 0

        return p

p = stacked_barplot(input_df = cf_df)
show(p)

In [99]:
p = stacked_barplot(input_df = sc_df)
show(p)

In [100]:
p = stacked_barplot(input_df = chembl_df)
show(p)

In [107]:
from bokeh.layouts import row
from bokeh.io import show

p1 = stacked_barplot(input_df=chembl_df, width=400)
p2 = stacked_barplot(input_df=cf_df, width=400)
p3 = stacked_barplot(input_df=sc_df, width=400)

layout = row(p1, p2, p3)
show(layout)


In [137]:
def summary_counts(df: pd.DataFrame):

    cellline = df['CellLine Count'].sum()
    tissue = df['Tissue Count'].sum()
    celltype = df['CellType Count'].sum()

    return (tissue, celltype, cellline)

chembl_counts = summary_counts(chembl_df)
cf_counts = summary_counts(cf_df)
sc_counts = summary_counts(sc_df)
print(sc_counts)


entity_types = ['Tissue', 'CellType', 'CellLine']
sources = ['ChEMBL', 'CellFinder', 'SingleCell']

x = [(source, entity_type) for source in sources for entity_type in entity_types] #Labels
counts = []
for i in [chembl_counts, cf_counts, sc_counts]:
    counts.append(i[0])
    counts.append(i[1])
    counts.append(i[2])

source = ColumnDataSource(data=dict(x=x, counts=counts))

p = figure(x_range=FactorRange(*x), height=300, width=550, title=f"Entity Counts",
           toolbar_location=None, tools="")

p.vbar(x='x', top='counts', width=0.9, source=source, fill_color=color_mapper, line_color=color_mapper)

p.y_range.start = 0
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None
p.xaxis.major_label_orientation = 1


show(p)


(np.float64(2385.0), np.float64(2752.0), np.float64(47.0))
